In [1]:
import os
import json
import requests
import pandas as pd
import plotly.express as px
import streamlit as st
from dotenv import load_dotenv

def load_and_validate_api_key():
    # load .env
    load_dotenv()

    # assign api_key from .env file
    api_key = os.getenv("FRED_API_KEY")

    # if api key incorrect or not found. raise error
    if not api_key:
        raise ValueError("FRED_API_KEY was not found! Check your .env file.")

    # else print success msg
    print("API Key loaded successfully (hidden for security).")

    return api_key


def prepare_end_point_parameters(api_key, series_id="CPIAUCSL"):
    # check if "data" folder exists, if not make it
    os.makedirs("data", exist_ok=True)
    file_path = f"data/{series_id}.json"

    # check if file exists locally in data folder
    if os.path.exists(file_path):
        print(f"Loading {series_id} data from local cache ('{file_path}')...")
        with open(file_path, "r", encoding="utf-8") as f:
            # associated json is loaded up and assigned
            cached_data = json.load(f)
        
        class MockResponse:
            def __init__(self, data):
                self.status_code = 200
                self._data = data
                self.text = ""
            def json(self):
                return self._data
                
        return MockResponse(cached_data)

    #if file doesnt exist, fetch from FRED API
    print(f"Fetching {series_id} data from FRED API...")
    url = "https://api.stlouisfed.org/fred/series/observations"

    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
    }

    response = requests.get(url, params=params, timeout=10)

    # Save the JSON response if successful
    if response.status_code == 200:
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(response.json(), f, indent=4)
        print(f"Successfully saved data to {file_path}")
    else:
        print(f"Error {response.status_code}: {response.text}")

    return response


def process_response(response, series_id):
    # status_code of 200 means request success
    if response.status_code == 200:
        data = response.json().get("observations", [])
        df = pd.DataFrame(data)[["date", "value"]]
        
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["date"] = pd.to_datetime(df["date"])
        df = df.dropna().sort_values("date")
        
        print("\nConnection Successful!")
        print(f"Fetched {len(df)} observations for {series_id}.")
        print("\nLatest 5 records:")
        print(df.tail(5))
    else:
        print(f"Error {response.status_code}: {response.text}")


if __name__ == "__main__":
    my_api_key = load_and_validate_api_key()
    target_series = "CPIAUCSL"

    api_response = prepare_end_point_parameters(my_api_key, target_series)
    process_response(api_response, target_series)

API Key loaded successfully (hidden for security).
Loading CPIAUCSL data from local cache ('data/CPIAUCSL.json')...

Connection Successful!
Fetched 955 observations for CPIAUCSL.

Latest 5 records:
          date    value
951 2026-04-01  332.407
952 2026-05-01  333.979
953 2026-06-01  332.568
954 2026-07-01  332.813
955 2026-08-01  334.131


In [2]:
def load_data(series_id: str = "CPIAUCSL") -> pd.DataFrame:
    file_path = f"data/{series_id}.json"

    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    return raw_data

In [3]:
df = load_data("CPIAUCSL")
list(df.keys())

['realtime_start',
 'realtime_end',
 'observation_start',
 'observation_end',
 'units',
 'output_type',
 'file_type',
 'order_by',
 'sort_order',
 'count',
 'offset',
 'limit',
 'observations']

In [4]:
temp_df = pd.DataFrame(df["observations"])[["date", "value"]]
temp_df.head()

,date,value
0,1947-01-01,21.48
1,1947-02-01,21.62
2,1947-03-01,22.0
3,1947-04-01,22.0
4,1947-05-01,21.95


In [5]:
st.title("CPI Overtime")

figure = px.line(
    temp_df,
    x="date",
    y="value",
    title="cpi overtime",
    markers=True
)

st.plotly_chart(figure, width="stretch")

2026-09-25 10:39:22.678 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 10:39:23.031 
  command:

    streamlit run c:\Users\valde\Desktop\unnamed_fred_data_analysis_app\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-25 10:39:23.033 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 10:39:23.033 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 10:39:23.130 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 10:39:23.130 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 10:39:23.132 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 1

DeltaGenerator()

In [ ]:
def display(df):
    st.title("CPI Overtime")

    # 1. Create the time-range selection buttons
    time_frame = st.segmented_control(
        "Select Time Horizon", options=["1Y", "5Y", "10Y", "Max"], default="10Y"
    )

    # 2. Filter the dataframe based on the selected button
    # Get the latest date available in your dataset as the anchor point
    max_date = temp_df["date"].max()

    if time_frame == "1Y":
        start_date = max_date - pd.DateOffset(years=1)
    elif time_frame == "5Y":
        start_date = max_date - pd.DateOffset(years=5)
    elif time_frame == "10Y":
        start_date = max_date - pd.DateOffset(years=10)
    else:  # "Max"
        start_date = temp_df["date"].min()

    # Apply the filter
    filtered_df = temp_df[temp_df["date"] >= start_date]

    # 3. Plot the filtered data
    figure = px.line(
        filtered_df,
        x="date",
        y="value",
        title=f"CPI Overtime ({time_frame})",
        markers=True,
    )

    st.plotly_chart(figure, width="stretch")

In [ ]:
def display(df):
    st.title("CPI Overtime")

    figure = px.line(
        df,
        x="date",
        y="value",
        title="cpi overtime",
        markers=True
    )

    st.plotly_chart(figure, width="stretch")